# Hãy nâng cấp lên trình độ CHUYÊN NGHIỆP!

Các kỹ thuật RAG nâng cao!

Hãy bắt đầu bằng cách tìm hiểu sâu về quá trình nạp dữ liệu:

1. Không dùng LangChain! Chỉ dùng các công cụ gốc để có độ linh hoạt tối đa
2. Hãy dùng một LLM để chia các đoạn một cách hợp lý
3. Hãy dùng kích thước đoạn và bộ mã hóa tốt nhất từ hôm qua
4. Hãy để LLM viết lại các đoạn theo cách hữu ích nhất ("tiền xử lý tài liệu")

In [53]:
from pathlib import Path  # Nhập Path từ gói pathlib.
from openai import OpenAI  # Nhập OpenAI từ gói openai.
from dotenv import load_dotenv  # Nhập load_dotenv từ gói dotenv.
from pydantic import BaseModel, Field  # Nhập BaseModel, Field từ gói pydantic.
from chromadb import PersistentClient  # Nhập PersistentClient từ gói chromadb.
from tqdm import tqdm  # Nhập tqdm từ gói tqdm.
from litellm import completion  # Nhập completion từ gói litellm.
import numpy as np  # Nạp numpy as np để sử dụng trong notebook.
import re  # Dùng để làm sạch JSON trả về từ mô hình.
from sklearn.manifold import TSNE  # Nhập TSNE từ gói sklearn.manifold.
import plotly.graph_objects as go  # Nạp plotly.graph_objects as go để sử dụng trong notebook.


load_dotenv(override=True)  # Nạp lại các biến trong tệp `.env` vào môi trường chạy.

MODEL = "gpt-4.1-nano"  # Chọn tên mô hình ngôn ngữ sẽ được sử dụng.

DB_NAME = "preprocessed_db"  # Đặt tên thư mục cơ sở dữ liệu vectơ.
collection_name = "docs"  # Đặt tên collection dùng để lưu các đoạn trong Chroma.
embedding_model = "text-embedding-3-large"  # Chọn mô hình dùng để tạo embedding.
KNOWLEDGE_BASE_PATH = Path("knowledge-base")  # Tạo đường dẫn tới thư mục cơ sở tri thức.
AVERAGE_CHUNK_SIZE = 500  # Đặt kích thước trung bình mục tiêu cho mỗi đoạn.

openai = OpenAI()  # Khởi tạo client OpenAI để gọi các API.

In [54]:
# Lấy cảm hứng từ Document của LangChain — hãy tạo một lớp tương tự

class Result(BaseModel):  # Khai báo mô hình dữ liệu cho một kết quả truy xuất cùng metadata.
    page_content: str  # Khai báo trường lưu nội dung văn bản của kết quả.
    metadata: dict  # Khai báo trường lưu metadata của kết quả.

In [55]:
# Một lớp biểu diễn đoạn văn bản một cách đầy đủ

class Chunk(BaseModel):  # Khai báo mô hình dữ liệu có cấu trúc cho một đoạn tài liệu.
    headline: str = Field(description="Tiêu đề ngắn gọn cho đoạn này, thường chỉ vài từ và có khả năng cao nhất xuất hiện trong một truy vấn")  # Khai báo trường tiêu đề ngắn cho một đoạn.
    summary: str = Field(description="Một vài câu tóm tắt nội dung của đoạn này để trả lời các câu hỏi thường gặp")  # Khai báo trường tóm tắt nội dung của một đoạn.
    original_text: str = Field(description="Nguyên văn của đoạn này từ tài liệu được cung cấp, giữ nguyên tuyệt đối, không thay đổi dưới bất kỳ hình thức nào")  # Khai báo trường lưu nguyên văn đoạn tài liệu.

    def as_result(self, document):  # Khai báo hàm `as_result` để đóng gói bước xử lý này.
        metadata = {"source": document["source"], "type": document["type"]}  # Tạo metadata nguồn và loại tài liệu cho kết quả.
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)  # Chuyển Chunk thành Result để lưu và truy xuất.


class Chunks(BaseModel):  # Khai báo mô hình bao bọc danh sách các đoạn do LLM trả về.
    chunks: list[Chunk]  # Khai báo trường chứa danh sách các đối tượng Chunk.

## Ba bước:

1. Lấy tài liệu từ cơ sở tri thức, giống như LangChain đã làm
2. Gọi một LLM để chuyển tài liệu thành các đoạn
3. Lưu các đoạn vào Chroma

Chỉ vậy thôi!

### Hãy bắt đầu với Bước 1

In [56]:
def fetch_documents():  # Khai báo hàm đọc toàn bộ tài liệu từ cơ sở tri thức.
    """Phiên bản tự xây dựng của LangChain DirectoryLoader"""  # Mô tả ngắn gọn mục đích và kết quả của hàm này.

    documents = []  # Khởi tạo danh sách chứa các tài liệu được nạp.

    for folder in KNOWLEDGE_BASE_PATH.iterdir():  # Lặp qua từng thư mục con của cơ sở tri thức.
        doc_type = folder.name  # Lấy loại tài liệu từ tên thư mục cha.
        for file in folder.rglob("*.md"):  # Lặp đệ quy qua từng tệp Markdown trong thư mục.
            with open(file, "r", encoding="utf-8") as f:  # Mở tệp hiện tại ở chế độ đọc văn bản UTF-8 và tự động đóng sau khi dùng.
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})  # Thêm tài liệu cùng loại, nguồn và nội dung vào danh sách.

    print(f"Đã nạp {len(documents)} tài liệu")  # In giá trị hoặc thông báo này ra phần output của ô.
    return documents  # Trả về danh sách tài liệu đã nạp.

In [57]:
documents = fetch_documents()  # Khởi tạo danh sách chứa các tài liệu được nạp.

Đã nạp 76 tài liệu


### Xong rồi! Chuyển sang Bước 2 — tạo các đoạn

In [58]:
def make_prompt(document):  # Khai báo hàm tạo prompt yêu cầu LLM chia một tài liệu thành các đoạn.
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1  # Ước tính số đoạn cần tạo từ độ dài tài liệu.
    return f"""
Bạn nhận một tài liệu và chia tài liệu đó thành các đoạn chồng lấn cho một Cơ sở tri thức.

Tài liệu đến từ ổ đĩa dùng chung của một công ty có tên Insurellm.
Loại tài liệu: {document["type"]}
Tài liệu được lấy từ: {document["source"]}

Một chatbot sẽ sử dụng các đoạn này để trả lời câu hỏi về công ty.
Bạn nên chia tài liệu theo cách phù hợp, đồng thời bảo đảm toàn bộ tài liệu được trả về trong các đoạn — không được bỏ sót bất kỳ nội dung nào.
Tài liệu này có lẽ nên được chia thành {how_many} đoạn, nhưng bạn có thể dùng nhiều hơn hoặc ít hơn nếu thấy phù hợp.
Các đoạn nên chồng lấn ở mức hợp lý; thông thường là khoảng 25% hoặc khoảng 50 từ, để cùng một nội dung xuất hiện trong nhiều đoạn nhằm mang lại kết quả truy xuất tốt nhất.

Với mỗi đoạn, bạn cần cung cấp một tiêu đề, một phần tóm tắt và nguyên văn của đoạn đó.
Khi kết hợp lại, các đoạn của bạn phải đại diện cho toàn bộ tài liệu và có phần chồng lấn.

Lưu ý quan trọng: chỉ trả về một đối tượng JSON hợp lệ theo schema sau, không thêm bất kỳ văn bản nào trước hoặc sau JSON:
{{
  "chunks": [
    {{
      "headline": "string",
      "summary": "string",
      "original_text": "string"
    }}
  ]
}}

Đây là tài liệu:

{document["text"]}

Hãy trả lời bằng JSON hợp lệ duy nhất.
"""  # Hoàn tất nội dung chuỗi nhiều dòng cho `chuỗi prompt`.


In [59]:
print(make_prompt(documents[0]))  # In giá trị hoặc thông báo này ra phần output của ô.


Bạn nhận một tài liệu và chia tài liệu đó thành các đoạn chồng lấn cho một Cơ sở tri thức.

Tài liệu đến từ ổ đĩa dùng chung của một công ty có tên Insurellm.
Loại tài liệu: company
Tài liệu được lấy từ: knowledge-base/company/about.md

Một chatbot sẽ sử dụng các đoạn này để trả lời câu hỏi về công ty.
Bạn nên chia tài liệu theo cách phù hợp, đồng thời bảo đảm toàn bộ tài liệu được trả về trong các đoạn — không được bỏ sót bất kỳ nội dung nào.
Tài liệu này có lẽ nên được chia thành 5 đoạn, nhưng bạn có thể dùng nhiều hơn hoặc ít hơn nếu thấy phù hợp.
Các đoạn nên chồng lấn ở mức hợp lý; thông thường là khoảng 25% hoặc khoảng 50 từ, để cùng một nội dung xuất hiện trong nhiều đoạn nhằm mang lại kết quả truy xuất tốt nhất.

Với mỗi đoạn, bạn cần cung cấp một tiêu đề, một phần tóm tắt và nguyên văn của đoạn đó.
Khi kết hợp lại, các đoạn của bạn phải đại diện cho toàn bộ tài liệu và có phần chồng lấn.

Lưu ý quan trọng: chỉ trả về một đối tượng JSON hợp lệ theo schema sau, không thêm bất k

In [60]:
def make_messages(document):  # Khai báo hàm đóng gói prompt thành danh sách tin nhắn cho API.
    return [  # Trả kết quả này về cho nơi gọi hàm.
        {"role": "user", "content": make_prompt(document)},  # Thêm tin nhắn người dùng vào danh sách tin nhắn.
    ]  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

In [61]:
make_messages(documents[0])  # Tạo và hiển thị danh sách tin nhắn cho tài liệu mẫu.

[{'role': 'user',
  'content': '\nBạn nhận một tài liệu và chia tài liệu đó thành các đoạn chồng lấn cho một Cơ sở tri thức.\n\nTài liệu đến từ ổ đĩa dùng chung của một công ty có tên Insurellm.\nLoại tài liệu: company\nTài liệu được lấy từ: knowledge-base/company/about.md\n\nMột chatbot sẽ sử dụng các đoạn này để trả lời câu hỏi về công ty.\nBạn nên chia tài liệu theo cách phù hợp, đồng thời bảo đảm toàn bộ tài liệu được trả về trong các đoạn — không được bỏ sót bất kỳ nội dung nào.\nTài liệu này có lẽ nên được chia thành 5 đoạn, nhưng bạn có thể dùng nhiều hơn hoặc ít hơn nếu thấy phù hợp.\nCác đoạn nên chồng lấn ở mức hợp lý; thông thường là khoảng 25% hoặc khoảng 50 từ, để cùng một nội dung xuất hiện trong nhiều đoạn nhằm mang lại kết quả truy xuất tốt nhất.\n\nVới mỗi đoạn, bạn cần cung cấp một tiêu đề, một phần tóm tắt và nguyên văn của đoạn đó.\nKhi kết hợp lại, các đoạn của bạn phải đại diện cho toàn bộ tài liệu và có phần chồng lấn.\n\nLưu ý quan trọng: chỉ trả về một đối tượn

In [62]:
def extract_json_from_text(text):
    if not text:
        return "{}"

    # Bỏ markdown code fence nếu có
    match = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        text = match.group(1)

    # Chỉ lấy phần JSON bắt đầu từ dấu { đầu tiên và kết thúc ở } cuối cùng
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]

    return text


def repair_truncated_json(text):
    """Cố gắng khép JSON nếu model trả về dữ liệu bị cắt ở cuối."""
    cleaned = text.strip()
    if not cleaned:
        return "{}"

    # Cố gắng đóng các ngoặc còn thiếu
    missing_closing_braces = cleaned.count("{") - cleaned.count("}")
    if missing_closing_braces > 0:
        cleaned += "}" * missing_closing_braces

    missing_closing_brackets = cleaned.count("[") - cleaned.count("]")
    if missing_closing_brackets > 0:
        cleaned += "]" * missing_closing_brackets

    return cleaned


def parse_chunks_response(reply):
    cleaned = extract_json_from_text(reply)
    cleaned = repair_truncated_json(cleaned)
    try:
        return Chunks.model_validate_json(cleaned).chunks
    except Exception:
        return []


def process_document(document):
    messages = make_messages(document)
    response = completion(
        model=MODEL,
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=4000,
    )
    reply = response.choices[0].message.content

    doc_as_chunks = parse_chunks_response(reply)
    if not doc_as_chunks:
        print("Bỏ qua tài liệu vì mô hình trả về JSON bị cắt hoặc không hợp lệ.")
        return []

    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [63]:
sample = '{"chunks":[{"headline":"A","summary":"B","original_text":"C"}]}'
cleaned = extract_json_from_text(f"```json\n{sample}\n```")
print(cleaned)
print(parse_chunks_response(cleaned)[0].headline)

{"chunks":[{"headline":"A","summary":"B","original_text":"C"}]}
A


In [64]:
# Cell này được giữ trống để tránh ghi đè lại hàm parse JSON đã được sửa ở cell trước.
# Các hàm xử lý JSON robust đang được định nghĩa trong cell #VSC-a45405d8.

In [65]:
process_document(documents[0])  # Xử lý tài liệu mẫu thành các đoạn có cấu trúc và hiển thị kết quả.

[Result(page_content='Giới thiệu về Insurellm\n\nInsurellm là một công ty công nghệ bảo hiểm được thành lập vào năm 2015 bởi Avery Lancaster, ban đầu tập trung vào các sản phẩm đổi mới trong ngành bảo hiểm.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Phát triển ban đầu và các sản phẩm chính\n\nTrong 5 năm đầu, Insurellm phát triển nhanh với các sản phẩm như Markellm, Carllm, Homellm, và Rellm, mở rộng quy mô và số lượng nhân viên đến 200 người với 12 văn phòng tại Mỹ.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insu

In [66]:
def create_chunks(documents):  # Khai báo hàm xử lý lần lượt mọi tài liệu để tạo danh sách đoạn.
    chunks = []  # Tạo hoặc lưu danh sách các đoạn tài liệu.
    for doc in tqdm(documents):  # Lặp qua tài liệu và hiển thị thanh tiến độ.
        chunks.extend(process_document(doc))  # Bổ sung các đoạn của tài liệu hiện tại vào danh sách chung.
    return chunks  # Trả về danh sách các đoạn đã tạo.

In [67]:
chunks = create_chunks(documents)  # Tạo hoặc lưu danh sách các đoạn tài liệu.

100%|██████████| 76/76 [11:29<00:00,  9.07s/it]


In [68]:
print(len(chunks))  # In giá trị hoặc thông báo này ra phần output của ô.

476


### Thật dễ dàng! Tuy có hơi chậm một chút.

Trong phiên bản mô-đun Python, tôi đã khéo léo dùng Pool đa tiến trình để chạy song song,
nhưng nếu bạn gặp Lỗi giới hạn tốc độ thì có thể tắt tính năng này trong mã.

### Cuối cùng, Bước 3 — lưu các embedding

In [69]:
def create_embeddings(chunks):  # Khai báo hàm tạo embedding và lưu các vectơ vào Chroma.
    chroma = PersistentClient(path=DB_NAME)  # Mở client Chroma bền vững tại thư mục cơ sở dữ liệu.
    if collection_name in [c.name for c in chroma.list_collections()]:  # Kiểm tra collection đích đã tồn tại trong Chroma hay chưa.
        chroma.delete_collection(collection_name)  # Xóa collection cũ để tránh trộn với dữ liệu của lần chạy trước.

    texts = [chunk.page_content for chunk in chunks]  # Trích nội dung văn bản của tất cả các đoạn.
    emb = openai.embeddings.create(model=embedding_model, input=texts).data  # Gọi API để tạo embedding cho danh sách văn bản.
    vectors = [e.embedding for e in emb]  # Chuyển danh sách embedding thành mảng NumPy.

    collection = chroma.get_or_create_collection(collection_name)  # Lấy đối tượng collection để đọc hoặc ghi dữ liệu Chroma.

    ids = [str(i) for i in range(len(chunks))]  # Tạo id duy nhất dạng chuỗi cho từng đoạn.
    metas = [chunk.metadata for chunk in chunks]  # Trích metadata của từng đoạn để lưu cùng vectơ.

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)  # Ghi id, vectơ, văn bản và metadata của các đoạn vào Chroma.
    print(f"Kho vectơ đã được tạo với {collection.count()} tài liệu")  # In giá trị hoặc thông báo này ra phần output của ô.

In [70]:
create_embeddings(chunks)  # Tạo embedding cho tất cả các đoạn và lưu chúng vào Chroma.

Kho vectơ đã được tạo với 476 tài liệu


# Không còn gì phải làm ở đây nữa... đúng không?

Khoan đã! Bạn nghĩ tôi quên rồi sao??

In [71]:
chroma = PersistentClient(path=DB_NAME)  # Mở client Chroma bền vững tại thư mục cơ sở dữ liệu.
collection = chroma.get_or_create_collection(collection_name)  # Lấy đối tượng collection để đọc hoặc ghi dữ liệu Chroma.
result = collection.get(include=['embeddings', 'documents', 'metadatas'])  # Lưu kết quả của bước xử lý hiện tại.
vectors = np.array(result['embeddings'])  # Chuyển danh sách embedding thành mảng NumPy.
documents = result['documents']  # Khởi tạo danh sách chứa các tài liệu được nạp.
metadatas = result['metadatas']  # Lấy metadata tương ứng với các tài liệu đã truy xuất.
doc_types = [metadata['type'] for metadata in metadatas]  # Trích loại tài liệu từ metadata để phân nhóm dữ liệu.
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]  # Ánh xạ mỗi loại tài liệu sang một màu trên biểu đồ.

In [72]:
tsne = TSNE(n_components=2, random_state=42)  # Khởi tạo t-SNE để giảm số chiều của vectơ.
reduced_vectors = tsne.fit_transform(vectors)  # Giảm các vectơ xuống số chiều cần trực quan hóa.

# Tạo biểu đồ phân tán 2D
fig = go.Figure(data=[go.Scatter(  # Tạo đối tượng biểu đồ Plotly.
    x=reduced_vectors[:, 0],  # Gán dữ liệu tọa độ cho trục x của biểu đồ.
    y=reduced_vectors[:, 1],  # Gán dữ liệu tọa độ cho trục y của biểu đồ.
    mode='markers',  # Chọn chế độ hiển thị bằng các điểm đánh dấu.
    marker=dict(size=5, color=colors, opacity=0.8),  # Cấu hình kích thước, màu và độ trong suốt của các điểm.
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],  # Tạo nội dung mô tả hiển thị khi rê chuột qua từng điểm.
    hoverinfo='text'  # Yêu cầu Plotly dùng phần văn bản làm thông tin khi rê chuột.
)])  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.update_layout(title='Trực quan hóa kho vectơ Chroma trong không gian 2D',  # Cấu hình tiêu đề, trục, kích thước và lề của biểu đồ.
    scene=dict(xaxis_title='x',yaxis_title='y'),  # Đặt nhãn cho các trục trong vùng biểu đồ.
    width=800,  # Đặt chiều rộng của biểu đồ theo pixel.
    height=600,  # Đặt chiều cao của biểu đồ theo pixel.
    margin=dict(r=20, b=10, l=10, t=40)  # Đặt khoảng lề xung quanh biểu đồ.
)  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.show()  # Hiển thị biểu đồ Plotly trong notebook.

In [73]:
tsne = TSNE(n_components=3, random_state=42)  # Khởi tạo t-SNE để giảm số chiều của vectơ.
reduced_vectors = tsne.fit_transform(vectors)  # Giảm các vectơ xuống số chiều cần trực quan hóa.

# Tạo biểu đồ phân tán 3D
fig = go.Figure(data=[go.Scatter3d(  # Tạo đối tượng biểu đồ Plotly.
    x=reduced_vectors[:, 0],  # Gán dữ liệu tọa độ cho trục x của biểu đồ.
    y=reduced_vectors[:, 1],  # Gán dữ liệu tọa độ cho trục y của biểu đồ.
    z=reduced_vectors[:, 2],  # Gán dữ liệu tọa độ cho trục z của biểu đồ.
    mode='markers',  # Chọn chế độ hiển thị bằng các điểm đánh dấu.
    marker=dict(size=5, color=colors, opacity=0.8),  # Cấu hình kích thước, màu và độ trong suốt của các điểm.
    text=[f"Loại: {t}<br>Văn bản: {d[:100]}..." for t, d in zip(doc_types, documents)],  # Tạo nội dung mô tả hiển thị khi rê chuột qua từng điểm.
    hoverinfo='text'  # Yêu cầu Plotly dùng phần văn bản làm thông tin khi rê chuột.
)])  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.update_layout(  # Cấu hình tiêu đề, trục, kích thước và lề của biểu đồ.
    title='Trực quan hóa kho vectơ Chroma trong không gian 3D',  # Đặt tiêu đề hiển thị cho biểu đồ.
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),  # Đặt nhãn cho các trục trong vùng biểu đồ.
    width=900,  # Đặt chiều rộng của biểu đồ theo pixel.
    height=700,  # Đặt chiều cao của biểu đồ theo pixel.
    margin=dict(r=10, b=10, l=10, t=40)  # Đặt khoảng lề xung quanh biểu đồ.
)  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

fig.show()  # Hiển thị biểu đồ Plotly trong notebook.

## Và bây giờ — hãy xây dựng một hệ thống RAG nâng cao!

Chúng ta sẽ sử dụng các kỹ thuật sau:

1. Xếp hạng lại — sắp xếp lại thứ tự các kết quả
2. Viết lại truy vấn

In [74]:
class RankOrder(BaseModel):  # Khai báo mô hình dữ liệu cho thứ tự id sau khi xếp hạng lại.
    order: list[int] = Field(  # Kiểm tra phản hồi JSON và lấy thứ tự id đã xếp hạng.
        description="Thứ tự liên quan của các đoạn, từ liên quan nhất đến ít liên quan nhất, dựa theo số id của đoạn"  # Mô tả ý nghĩa của trường để LLM tạo dữ liệu đúng cấu trúc.
    )  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.

In [75]:
def rerank(question, chunks):  # Khai báo hàm dùng LLM để xếp hạng lại các đoạn theo mức độ liên quan.
    system_prompt = """
Bạn là một hệ thống xếp hạng lại tài liệu.
Bạn được cung cấp một câu hỏi và danh sách các đoạn văn bản liên quan từ kết quả truy vấn một cơ sở tri thức.
Các đoạn được cung cấp theo thứ tự truy xuất; thứ tự này gần đúng theo mức độ liên quan, nhưng bạn có thể cải thiện nó.
Bạn phải xếp hạng các đoạn được cung cấp theo mức độ liên quan đến câu hỏi, với đoạn liên quan nhất đứng đầu.
Chỉ trả lời bằng danh sách id của các đoạn theo thứ tự đã xếp hạng, không thêm bất kỳ nội dung nào khác. Phải bao gồm tất cả id của các đoạn được cung cấp sau khi xếp hạng lại.
"""  # Hoàn tất nội dung chuỗi nhiều dòng cho `system_prompt`.
    user_prompt = f"Người dùng đã hỏi câu hỏi sau:\n\n{question}\n\nHãy sắp xếp tất cả các đoạn văn bản theo mức độ liên quan đến câu hỏi, từ liên quan nhất đến ít liên quan nhất. Phải bao gồm tất cả id của các đoạn được cung cấp sau khi xếp hạng lại.\n\n"  # Khởi tạo prompt người dùng với câu hỏi và yêu cầu xếp hạng.
    user_prompt += "Đây là các đoạn:\n\n"  # Nối thêm nội dung cần thiết vào prompt người dùng.
    for index, chunk in enumerate(chunks):  # Duyệt các đoạn cùng chỉ số để tạo id trong prompt.
        user_prompt += f"# ID ĐOẠN: {index + 1}:\n\n{chunk.page_content}\n\n"  # Nối thêm nội dung cần thiết vào prompt người dùng.
    user_prompt += "Chỉ trả lời bằng danh sách id của các đoạn theo thứ tự đã xếp hạng, không thêm bất kỳ nội dung nào khác."  # Nối thêm nội dung cần thiết vào prompt người dùng.
    messages = [  # Tạo danh sách tin nhắn theo định dạng mà mô hình yêu cầu.
        {"role": "system", "content": system_prompt},  # Thêm prompt hệ thống vào danh sách tin nhắn.
        {"role": "user", "content": user_prompt},  # Thêm tin nhắn người dùng vào danh sách tin nhắn.
    ]  # Đóng cấu trúc dữ liệu hoặc lời gọi hàm nhiều dòng.
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)  # Gọi mô hình và lưu phản hồi trả về.
    reply = response.choices[0].message.content  # Lấy phần nội dung văn bản từ phản hồi của mô hình.
    order = RankOrder.model_validate_json(reply).order  # Kiểm tra phản hồi JSON và lấy thứ tự id đã xếp hạng.
    print(order)  # Hiển thị thứ tự id sau khi xếp hạng lại để kiểm tra.
    return [chunks[i - 1] for i in order]  # Sắp xếp lại các đoạn theo thứ tự id mà mô hình trả về.

In [76]:
RETRIEVAL_K = 10  # Đặt số lượng kết quả gần nhất cần truy xuất.

def fetch_context_unranked(question):  # Khai báo hàm truy xuất các đoạn gần nhất trước khi xếp hạng lại.
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding  # Tạo hoặc lưu truy vấn dùng để tìm kiếm trong cơ sở tri thức.
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)  # Truy vấn Chroma để lấy các đoạn gần nhất với embedding câu hỏi.
    chunks = []  # Tạo hoặc lưu danh sách các đoạn tài liệu.
    for result in zip(results["documents"][0], results["metadatas"][0]):  # Ghép từng văn bản truy xuất với metadata tương ứng.
        chunks.append(Result(page_content=result[0], metadata=result[1]))  # Chuyển kết quả truy vấn thành đối tượng Result rồi thêm vào danh sách đoạn.
    return chunks  # Trả về danh sách các đoạn đã tạo.

In [77]:
question = "Ai đã giành giải IIOTY?"  # Đặt câu hỏi mẫu dùng để thử quy trình RAG.
chunks = fetch_context_unranked(question)  # Tạo hoặc lưu danh sách các đoạn tài liệu.

In [78]:
for chunk in chunks:  # Lặp qua từng đoạn để hiển thị nội dung rút gọn.
    print(chunk.page_content[:15]+"...")  # In phần đầu nội dung đoạn để xem nhanh thứ tự kết quả.

Tiến trình nghề...
Giải thưởng, ch...
Lịch sử đánh gi...
Lịch sử đánh gi...
Lịch sử hiệu su...
Nhận xét tổng q...
Lịch sử hiệu su...
Tổng kết về sự ...
Tiến trình nghề...
Hiệu suất làm v...


In [79]:
reranked = rerank(question, chunks)  # Lưu danh sách đoạn sau khi đã xếp hạng lại.

[1, 2, 10, 3, 4, 5, 6, 7, 8, 9]


In [80]:
for chunk in reranked:  # Lặp qua từng phần tử của tập dữ liệu này.
    print(chunk.page_content[:15]+"...")  # In phần đầu nội dung đoạn để xem nhanh thứ tự kết quả.

Tiến trình nghề...
Giải thưởng, ch...
Hiệu suất làm v...
Lịch sử đánh gi...
Lịch sử đánh gi...
Lịch sử hiệu su...
Nhận xét tổng q...
Lịch sử hiệu su...
Tổng kết về sự ...
Tiến trình nghề...


In [81]:
question = "Ai đã học tại Đại học Manchester?"  # Đặt câu hỏi mẫu dùng để thử quy trình RAG.
RETRIEVAL_K = 20  # Đặt số lượng kết quả gần nhất cần truy xuất.
chunks = fetch_context_unranked(question)  # Tạo hoặc lưu danh sách các đoạn tài liệu.
for index, c in enumerate(chunks):  # Duyệt các đoạn cùng chỉ số để tìm kết quả mong muốn.
    if "manchester" in c.page_content.lower():  # Kiểm tra đoạn hiện tại có nhắc đến Manchester hay không.
        print(index)  # In giá trị hoặc thông báo này ra phần output của ô.

In [82]:
reranked = rerank(question, chunks)  # Lưu danh sách đoạn sau khi đã xếp hạng lại.

[1, 2, 11, 13, 3, 4, 7, 17, 20, 19, 8, 9, 12, 14, 15, 16, 18]


In [83]:
for index, c in enumerate(reranked):  # Duyệt các đoạn cùng chỉ số để tìm kết quả mong muốn.
    if "manchester" in c.page_content.lower():  # Kiểm tra đoạn hiện tại có nhắc đến Manchester hay không.
        print(index)  # In giá trị hoặc thông báo này ra phần output của ô.

In [84]:
reranked[0].page_content  # Hiển thị nội dung của đoạn được xếp hạng liên quan nhất.

'Trình độ học vấn và kỹ năng chuyên môn\n\nRobert Chen có bằng thạc sĩ về Khoa học Máy tính từ Stanford và bằng cử nhân Kỹ thuật Máy tính từ MIT, cùng với kỹ năng chuyên sâu về các công nghệ như React, Node.js, TypeScript, PostgreSQL, AWS và kiến trúc microservices.\n\n- **Education:** MS in Computer Science from Stanford University, BS in Computer Engineering from MIT\n- **Technical Expertise:** Expert in React, Node.js, TypeScript, PostgreSQL, AWS, microservices architecture'

In [85]:
def fetch_context(question):  # Khai báo hàm truy xuất rồi xếp hạng lại ngữ cảnh.
    chunks = fetch_context_unranked(question)  # Tạo hoặc lưu danh sách các đoạn tài liệu.
    return rerank(question, chunks)  # Trả về kết quả truy xuất sau khi xếp hạng lại.

In [86]:
SYSTEM_PROMPT = """
Bạn là một trợ lý am hiểu và thân thiện, đại diện cho công ty Insurellm.
Bạn đang trò chuyện với người dùng về Insurellm.
Câu trả lời của bạn sẽ được đánh giá về độ chính xác, mức độ liên quan và tính đầy đủ, vì vậy hãy bảo đảm câu trả lời chỉ tập trung vào câu hỏi và trả lời câu hỏi một cách trọn vẹn.
Nếu không biết câu trả lời, hãy nói rõ điều đó.
Để làm ngữ cảnh, dưới đây là các trích đoạn cụ thể từ Cơ sở tri thức có thể liên quan trực tiếp đến câu hỏi của người dùng:
{context}

Dựa trên ngữ cảnh này, hãy trả lời câu hỏi của người dùng. Hãy chính xác, liên quan và đầy đủ.
"""  # Hoàn tất nội dung chuỗi nhiều dòng cho `SYSTEM_PROMPT`.

In [87]:
# Trong ngữ cảnh, hãy bao gồm nguồn của đoạn

def make_rag_messages(question, history, chunks):  # Khai báo hàm tạo chuỗi tin nhắn đầy đủ cho bước sinh câu trả lời RAG.
    context = "\n\n".join(f"Trích từ {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)  # Ghép nội dung các tài liệu truy xuất thành một chuỗi ngữ cảnh.
    system_prompt = SYSTEM_PROMPT.format(context=context)  # Tạo prompt hệ thống hoàn chỉnh cho lần gọi mô hình này.
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]  # Trả kết quả này về cho nơi gọi hàm.

In [88]:
def rewrite_query(question, history=[]):  # Khai báo hàm viết lại câu hỏi thành truy vấn tìm kiếm cụ thể hơn.
    """Viết lại câu hỏi của người dùng thành một câu hỏi cụ thể hơn, có khả năng tìm ra nội dung liên quan trong Cơ sở tri thức cao hơn."""  # Mô tả ngắn gọn mục đích và kết quả của hàm này.
    message = f"""
Bạn đang trò chuyện với một người dùng và trả lời các câu hỏi về công ty Insurellm.
Bạn sắp tra cứu thông tin trong một Cơ sở tri thức để trả lời câu hỏi của người dùng.

Đây là lịch sử cuộc trò chuyện của bạn với người dùng cho đến thời điểm hiện tại:
{history}

Và đây là câu hỏi hiện tại của người dùng:
{question}

Chỉ trả lời bằng một câu hỏi duy nhất đã được tinh chỉnh để bạn dùng khi tìm kiếm trong Cơ sở tri thức.
Đó phải là một câu hỏi cụ thể, RẤT ngắn gọn và có khả năng tìm ra nội dung liên quan cao nhất. Hãy tập trung vào các chi tiết của câu hỏi.
Không đề cập tên công ty trừ khi đó là một câu hỏi chung về công ty.
QUAN TRỌNG: CHỈ trả lời bằng truy vấn dành cho cơ sở tri thức, không thêm bất kỳ nội dung nào khác.
"""  # Hoàn tất nội dung chuỗi nhiều dòng cho `message`.
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])  # Gọi mô hình và lưu phản hồi trả về.
    return response.choices[0].message.content  # Trả về nội dung câu trả lời đầu tiên của mô hình.

In [89]:
rewrite_query("Ai đã giành giải IIOTY?", [])  # Viết lại câu hỏi mẫu thành truy vấn tìm kiếm và hiển thị kết quả.

'Ai đã nhận giải thưởng IIOTY?'

In [90]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:  # Khai báo hàm trả lời câu hỏi bằng quy trình RAG.
    """
    Trả lời câu hỏi bằng RAG, đồng thời trả về câu trả lời và ngữ cảnh đã truy xuất
    """  # Hoàn tất nội dung chuỗi nhiều dòng cho `chuỗi prompt`.
    query = rewrite_query(question, history)  # Tạo hoặc lưu truy vấn dùng để tìm kiếm trong cơ sở tri thức.
    print(query)  # Hiển thị truy vấn đã viết lại để kiểm tra.
    chunks = fetch_context(query)  # Tạo hoặc lưu danh sách các đoạn tài liệu.
    messages = make_rag_messages(question, history, chunks)  # Tạo danh sách tin nhắn theo định dạng mà mô hình yêu cầu.
    response = completion(model=MODEL, messages=messages)  # Gọi mô hình và lưu phản hồi trả về.
    return response.choices[0].message.content, chunks  # Trả về cả câu trả lời của mô hình và các đoạn ngữ cảnh đã dùng.

In [91]:
answer_question("Ai đã giành giải IIOTY?", [])  # Chạy toàn bộ quy trình RAG cho câu hỏi mẫu và hiển thị câu trả lời.

Giải IIOTY đã được trao cho ai?
[2, 1, 3, 6, 10, 12, 7, 8, 11, 9, 4, 14, 13, 16, 15, 17, 20, 18, 5, 7]


('Người đã giành giải thưởng Insurellm Innovator of the Year (IIOTY) là Maxine Thompson vào năm 2023.',
 [Result(page_content='Tiến trình nghề nghiệp của Maxine tại Insurellm (Giai đoạn 2021 đến nay)\n\nTừ tháng 1 năm 2021, Maxine được thăng chức lên Senior Data Engineer, dẫn dắt các dự án cải thiện thời gian truy xuất dữ liệu và trở thành mentor cho các kỹ sư trẻ, nhận giải thưởng Innovator của năm 2023.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.', metadata={'source': 'knowledge-base/employees/Maxine Thompson.md', 'type': 'employees'}),
  Result(page_content="Giải thưởng, chứng nhận và vai trò 

In [92]:
answer_question("Ai đã học tại Đại học Manchester?", [])  # Chạy toàn bộ quy trình RAG cho câu hỏi mẫu và hiển thị câu trả lời.

Ai đã học tại Đại học Manchester?
[1, 2, 3, 4, 7, 11, 13, 16, 18, 10, 6, 8, 12, 14, 5, 15, 17, 9, 20, 19]


('Xin lỗi, trong dữ liệu của tôi không có thông tin về ai đã học tại Đại học Manchester.',
 [Result(page_content='Trình độ học vấn và kỹ năng chuyên môn\n\nRobert Chen có bằng thạc sĩ về Khoa học Máy tính từ Stanford và bằng cử nhân Kỹ thuật Máy tính từ MIT, cùng với kỹ năng chuyên sâu về các công nghệ như React, Node.js, TypeScript, PostgreSQL, AWS và kiến trúc microservices.\n\n- **Education:** MS in Computer Science from Stanford University, BS in Computer Engineering from MIT\n- **Technical Expertise:** Expert in React, Node.js, TypeScript, PostgreSQL, AWS, microservices architecture', metadata={'source': 'knowledge-base/employees/Robert Chen.md', 'type': 'employees'}),
  Result(page_content='Học vấn, công bố và các hoạt động khác\n\nPriya Sharma có bằng PhD về Machine Learning từ Stanford, đã công bố hơn 7 bài báo, thường tham gia diễn thuyết tại các hội nghị ML và InsurTech, đồng sáng chế 2 bằng sáng chế liên quan đến ML trong bảo hiểm.\n\n## Other HR Notes\n- **Education:** PhD 